In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.runnables import RunnableConfig

In [3]:
checkpointer = InMemorySaver()

In [4]:
agent = create_agent(
    model="gpt-5-nano",
    tools=[],
    middleware=[
        SummarizationMiddleware(
            model="gpt-5-mini", trigger=("tokens", 100), keep=("messages", 2)
        )
    ],
    checkpointer=checkpointer,
)

In [5]:
config: RunnableConfig = {"configurable": {"thread_id": "hanhokim"}}

agent.invoke(
    {"messages": [{"role": "user", "content": "안녕 내이름은 김한호야.."}]}, config
)

{'messages': [HumanMessage(content='안녕 내이름은 김한호야..', additional_kwargs={}, response_metadata={}, id='cd1cd3dd-0b12-422f-868c-557e08bcd911'),
  AIMessage(content='반갑습니다, 김한호님! 이 대화에서 그렇게 불러도 될까요? 무엇을 도와드릴까요? 예를 들면:\n- 한국어 공부 팁이나 연습 문제\n- 코딩이나 기술 관련 질문\n- 글쓰기나 번역 도움\n- 아이디어 정리나 계획 세우기\n- 여행, 음식, 취미 추천\n\n원하시는 주제나 목표를 알려주시면 바로 도와드리겠습니다.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 555, 'prompt_tokens': 17, 'total_tokens': 572, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 448, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-DZ8Q9IACUgmGTGFm499F1v8L3c0ck', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019dcd52-5d2e-7ad2-92ae-ba83ab502c86-0', tool_calls=[], invalid_tool_calls=[], usage_met

In [6]:
agent.invoke(
    {"messages": [{"role": "user", "content": "난 인공지능을 배우고 있어"}]}, config
)

{'messages': [HumanMessage(content='Here is a summary of the conversation to date:\n\n## SESSION INTENT\nThe user greeted the assistant and introduced themselves ("안녕 내이름은 김한호야."). Primary goal: begin a conversation and provide their name (Kim Han-ho). Likely intent is to start interaction and enable the assistant to address them by name.\n\n## SUMMARY\n- User message (Korean): "안녕 내이름은 김한호야.." — greeting and self-introduction.\n- No questions, requests, or tasks were specified beyond the introduction.\n- No decisions, alternatives, or strategies were discussed or rejected.\n\n## ARTIFACTS\nNone\n\n## NEXT STEPS\n- Respond with an appropriate greeting in Korean and acknowledge the user\'s name (김한호).\n- Ask how to assist or what the user would like to do next.\n- Optionally confirm preferred level of formality (존댓말/반말) and preferred language for the conversation.', additional_kwargs={'lc_source': 'summarization'}, response_metadata={}, id='ee6090be-6939-4a12-9259-3f6dbe5620ec'),
  AIMe

In [7]:
agent.invoke(
    {"messages": [{"role": "user", "content": "넌 인공지능의 미래가 뭐라고 생각해?"}]},
    config,
)

{'messages': [HumanMessage(content='Here is a summary of the conversation to date:\n\n## SESSION INTENT\nThe user introduced themself (김한호) and wants to begin a conversation. Current explicit goal: the user is learning artificial intelligence and likely seeks help, guidance, or resources about AI.\n\n## SUMMARY\n- User: "안녕 내이름은 김한호야." — greeting and self-introduction (name: Kim Han-ho).\n- Assistant: Greeted the user in Korean, asked permission to use that name, offered example topics the assistant can help with (Korean study, coding/tech questions, writing/translation, idea organization/planning, travel/food/hobby recommendations), and asked about preferred level of formality and language.\n- User: "난 인공지능을 배우고 있어." — stated they are learning AI.\n- No detailed questions, goals, or background about their AI study were provided (no stated current level, topics of interest, timeframe, or preferred language/formality).\n\n## ARTIFACTS\nNone\n\n## NEXT STEPS\n- Greet the user and acknowl

In [8]:
# HITL
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware, wrap_tool_call
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command
from langchain.tools import tool

In [9]:
@tool
def search_docs(query: str) -> str:
    """문서를 검색합니다."""
    return f"[search] {query}검색 결과입니다"


@tool
def delete_file(filename: str) -> str:
    """파일을 삭제합니다."""

    return f"[delete] {filename}을 삭제 했습니다."

In [10]:
@wrap_tool_call
def log_tool_calls(request, handler):

    print(f"[TOOL START]  {request.tool.name} ")

    tool_call = getattr(request, "tool_call", None)
    if tool_call and "args" in tool_call and tool_call["args"] is not None:
        print(f"[TOOL ARGS] {tool_call['args']}")

    try:
        result = handler(request)

        print(f"[TOOL END]  result : {result}")
        return result

    except Exception as e:
        print(f"[TOOL ERROR] error = {e}")
        raise

In [11]:
checkpointer = InMemorySaver()

In [12]:
agent = create_agent(
    model="gpt-5-nano",
    tools=[search_docs, delete_file],
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "delete_file": True,
            },
            description_prefix="검토 필요 작업",
        ),
        log_tool_calls,
    ],
    checkpointer=checkpointer,
)

config = {"configurable": {"thread_id": "hanho1"}}

In [13]:
result = agent.invoke(
    {"messages": [{"role": "user", "content": "123.pdf 파일을 삭제해주세요."}]},
    config=config,
)

result

{'messages': [HumanMessage(content='123.pdf 파일을 삭제해주세요.', additional_kwargs={}, response_metadata={}, id='15ed624e-f29d-4007-aa84-38f7edb814eb'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 216, 'prompt_tokens': 149, 'total_tokens': 365, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 192, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-DZ8t13Gxe2NgBO23mEjxx6COMtcVD', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019dcd6d-ae08-7f42-b452-1c4511e0b317-0', tool_calls=[{'name': 'delete_file', 'args': {'filename': '123.pdf'}, 'id': 'call_SKInk7aBDBpQdXQo7uX7YAuU', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 149, 'output_tokens': 

In [14]:
resume_result = agent.invoke(
    Command(resume={"decisions": [{"type": "approve"}]}), config=config, version="v2"
)

[TOOL START]  delete_file 
[TOOL ARGS] {'filename': '123.pdf'}
[TOOL END]  result : content='[delete] 123.pdf을 삭제 했습니다.' name='delete_file' tool_call_id='call_SKInk7aBDBpQdXQo7uX7YAuU'


In [15]:
resume_result["messages"][-1].content

C:\Users\user\AppData\Local\Temp\ipykernel_36524\3022618795.py:1: LangGraphDeprecatedSinceV11: Accessing GraphOutput via `result[key]` is deprecated. Use `result.value` to access the output value directly, or `result.interrupts` for interrupts. Deprecated in LangGraph V1.1 to be removed in V3.0.
  resume_result["messages"][-1].content


'123.pdf 파일을 삭제했습니다. 다른 파일 정리나 추가로 도와드릴 작업이 있으면 말씀해 주세요.'

In [16]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware, wrap_tool_call
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command
from langchain.tools import tool
from pydantic import BaseModel, Field

In [17]:
@tool
def delete_file(filename: str) -> str:
    """문서를  삭제 합니다."""
    return f"[delete] {filename}을 삭제 하였습니다."

In [18]:
class ApprovalDecision(BaseModel):
    approved: bool = Field(
        description="사용자의 발화가 삭제 승인이면  True, 아니면  False"
    )
    reason: str = Field(description="판단이유를 짧게 설명")

In [19]:
@wrap_tool_call
def log_tool_calls(request, handler):

    print(f"[TOOL START]  {request.tool.name} ")

    tool_call = getattr(request, "tool_call", None)
    if tool_call and "args" in tool_call and tool_call["args"] is not None:
        print(f"[TOOL ARGS] {tool_call['args']}")

    try:
        result = handler(request)

        print(f"[TOOL END]  result : {result}")
        return result

    except Exception as e:
        print(f"[TOOL ERROR] error = {e}")
        raise

In [20]:
approval_agent = create_agent(
    model="gpt-5-nano",
    tools=[],
    system_prompt=(
        '당신은 사용자의 한국어 답변이 "파일을 삭제 승인"인지를 판단하는 판별기입니다.'
        "파일을 삭제하는데 명시적으로 동의한 경우에만 approved=True로 판단하세요"
        "사용자의 발화가 애매하면  False 로 판단하세요"
        '발화의 예: "승인","허가합니다.","네","진행하세요","삭제하세요", 등의 문구는 승인으로 볼수 있습니다.'
        "단순 질문,머뭇거림,불명확한 표현등은 승인으로 보면 안됩니다."
    ),
    response_format=ApprovalDecision,
)

In [21]:
checkpointer = InMemorySaver()

In [22]:
agent = create_agent(
    model="gpt-5-nano",
    tools=[delete_file],
    middleware=[
        log_tool_calls,
        HumanInTheLoopMiddleware(
            interrupt_on={
                "delete_file": {
                    "allowed_decisions": ["approve", "reject"],
                    "description": "파일 삭제는 사람이 승인을 한후에만 진행됩니다.",
                },
            },
            description_prefix="사람의 검토가 필요한 작업",
        ),
    ],
    checkpointer=checkpointer,
)

config = {"configurable": {"thread_id": "hanho2"}}

In [23]:
def request_delete(prompt: str):
    result = agent.invoke(
        {"messages": [{"role": "user", "content": prompt}]}, config=config, version="v2"
    )

    print(result)

    interrupts = result.interrupts
    messages = result.value["messages"]

    if interrupts:
        print("삭제를 하려면 승인여부를 입력해주세요 (예: 삭제해줘)")
        return "WAITING_APPROVAL"

    print(result["messages"][-1].content)
    return "DONE"

In [24]:
def handdler_followup_with_llm(prompt: str):

    judge_result = approval_agent.invoke(
        {
            "messages": [
                {
                    "role": "user",
                    "content": f"다음 사용자 발화가 삭제를 승인하는 뜻인지 판정하세요 사용자발화 : {prompt}",
                }
            ]
        }
    )

    dicision = judge_result["structured_response"]

    print(f"판정결과 - approved : {dicision.approved}    -reason : {dicision.reason}")

    if dicision.approved == True:
        resume_result = agent.invoke(
            Command(resume={"decisions": [{"type": "approve"}]}),
            config=config,
            version="v2",
        )
        print(resume_result["messages"][-1].content)
        return "APPROVED"

    else:
        resume_result = agent.invoke(
            Command(
                resume={
                    "decisions": [{"type": "reject", "reason": f"{dicision.reason}"}]
                }
            ),
            config=config,
            version="v2",
        )
        print(resume_result["messages"][-1].content)
        return "REJECT"

In [25]:
result = request_delete("a.pdf 파일을 삭제해주세요")

if result == "WAITING_APPROVAL":
    input_prompt = input("사용자 입력 : ")
    handdler_followup_with_llm(input_prompt)

GraphOutput(value={'messages': [HumanMessage(content='a.pdf 파일을 삭제해주세요', additional_kwargs={}, response_metadata={}, id='fc6bd121-f12f-486d-8d6e-de641ee53ee6'), AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 216, 'prompt_tokens': 130, 'total_tokens': 346, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 192, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-DZ9sTNIFcvZAlAk7iMShNbTRYUweW', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019dcda7-d0e5-7163-b21e-2a2fab37d516-0', tool_calls=[{'name': 'delete_file', 'args': {'filename': 'a.pdf'}, 'id': 'call_WEYZdxGWThwgX9HGat0cw45k', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 130, 'outpu

C:\Users\user\AppData\Local\Temp\ipykernel_36524\2811585200.py:24: LangGraphDeprecatedSinceV11: Accessing GraphOutput via `result[key]` is deprecated. Use `result.value` to access the output value directly, or `result.interrupts` for interrupts. Deprecated in LangGraph V1.1 to be removed in V3.0.
  print(resume_result["messages"][-1].content)


In [26]:
thread_id = "thread-1"
checkpointer.delete_thread(thread_id)

In [27]:
from langchain.agents import create_agent
from langchain.agents.middleware import (
    ToolCallLimitMiddleware,
    wrap_tool_call,
    ModelFallbackMiddleware,
    PIIMiddleware,
)
from langgraph.checkpoint.memory import InMemorySaver
from langchain.tools import tool
from langchain_google_genai import ChatGoogleGenerativeAI

In [28]:
gemini_model = ChatGoogleGenerativeAI(model="gemini-2.5-flash")

In [29]:
@tool
def search_docs(query: str) -> str:
    """문서를 검색합니다."""
    return f"[search] {query}문서를 검색하였습니다."


@tool
def make_image(query: str):
    """이미지를 생성합니다."""
    return f"{query} 이미지를 생성하였습니다."

In [30]:
@wrap_tool_call
def log_tool_calls(request, handler):

    print(f"[TOOL START]  {request.tool.name} ")

    tool_call = getattr(request, "tool_call", None)
    if tool_call and "args" in tool_call and tool_call["args"] is not None:
        print(f"[TOOL ARGS] {tool_call['args']}")

    try:
        result = handler(request)

        print(f"[TOOL END]  result : {result}")
        return result

    except Exception as e:
        print(f"[TOOL ERROR] error = {e}")
        raise

In [31]:
checkpointer = InMemorySaver()

agent = create_agent(
    model="gpt-5-nano",
    tools=[search_docs, make_image],
    middleware=[
        log_tool_calls,
        ToolCallLimitMiddleware(run_limit=3),
        ToolCallLimitMiddleware(
            tool_name="make_image",
            thread_limit=5,
            run_limit=1,
        ),
        PIIMiddleware("email", strategy="mask", apply_to_input=True),
        ModelFallbackMiddleware("gpt-5-nano", gemini_model),
    ],
    checkpointer=checkpointer,
)

config = {"configurable": {"thread_id": "hanhokim"}}

In [32]:
result = agent.invoke(
    {
        "messages": [
            {"role": "user", "content": "hanhokim@gmail.com 으로 메일을 전송해줘"}
        ]
    },
    config=config,
)

result

{'messages': [HumanMessage(content='hanhokim@****.com 으로 메일을 전송해줘', additional_kwargs={}, response_metadata={}, id='55fe7f67-58bd-4023-824e-ad5309fcb466'),
  AIMessage(content='도와드릴 수는 있지만, 제 시스템에서 직접 메일을 보내는 기능은 없어요. 대신 받을 사람에게 보낼 내용 작성은 도와드리고, 보내는 방법까지 안내해드릴게요.\n\n필요한 정보\n- 받는 사람 주소: 정확한 주소를 알려주실 수 있나요? 현재 주소는 마스킹되어 있어요.\n- 제목(Subject)\n- 본문 내용(한국어/영어 중 어떤 톤으로 원하나요? 포멀/캐주얼?)\n- 첨부파일 여부 및 파일 경로\n- 참조/참조(CC/BCC) 여부\n- 발신인 정보(이메일 주소, 이름, 소속 등)\n- 보내는 시점이나 긴급도(예: 지금 보냄 vs 특정 시간에 보내기)\n\n원하시는 경우 바로 사용 가능한 초안\n다음 중 하나를 선택해 주시면 바로 맞춤 초안을 만들어 드릴게요. 필요하면 톤이나 길이도 맞춰 드립니다.\n\n템플릿 A: 일반 비즈니스 문의 (폼 격식)\nSubject: [주제 요약]에 대한 확인 요청\nBody:\n안녕하세요, [이름]님.\n[상황/맥락 간단히 설명].\n문의 사항:\n- [질문 1]\n- [질문 2]\n첨부 자료: [첨부가 있다면 목록]\n회신 주시면 감사하겠습니다.\n감사합니다.\n[발신인 이름]\n[직책/소속]\n[연락처]\n\n템플릿 B: 회의 초대/일정 안내\nSubject: [회의 제목] 일정 공유 및 참석 확인 요청\nBody:\n안녕하세요, [이름]님.\n다음 일정으로 회의를 계획하고 있습니다.\n- 일시: [날짜] [시간]\n- 장소/링크: [장소 또는 화상회의 링크]\n참석 가능 여부를 알려주시면 준비에 도움이 되겠습니다.\n감사합니다.\n[발신인 이름]\n[직책/소속]\n[연락처]\n\n템플릿 C: 간단 확인 메일\nSu

In [33]:
pii_agent = create_agent(
    model="gpt-5-nano",
    tools=[],
    system_prompt=(
        "당신은 프롬프트를 받아서 프롬프트안에 개인정보가 들어있는지 판별하는 판별기입니다."
        "사용자의 이름이나 나이, 성별, 전화번호 등의 데이터가 들어오면  *****로 처리를 해주세요"
    ),
)

pii_agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "이름이 김한호이고, 성별은 남자이고, 전화번호는 010-1234-1234입니다. 위 정보를 요약해주세요",
            }
        ]
    }
)

{'messages': [HumanMessage(content='이름이 김한호이고, 성별은 남자이고, 전화번호는 010-1234-1234입니다. 위 정보를 요약해주세요', additional_kwargs={}, response_metadata={}, id='485fca33-4af2-485a-819e-f1c1198e8d0c'),
  AIMessage(content='요약(개인정보 가림):\n- 이름: *****\n- 성별: *****\n- 전화번호: *****', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 802, 'prompt_tokens': 97, 'total_tokens': 899, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 768, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-DZ9yT1E1qIXbSF1QlH3J5nJdXihYI', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019dcdad-806c-78a2-a0c7-b8218b67bf3d-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 97, 'output_tokens': 802, 'total_tokens': 899, 'input_